# 🏆 S6E9 Rebuilt Anchor — Tuned LGBM × 3 Seeds × Original Data

The gold-zone rebuild: everything we learned about this competition in one notebook.

**Key ingredients:**
1. **Original data integration** — the 10,000 real rows behind the synthetic generation, upweighted ×20 via sample weights
2. **DGP-aware features** — the buy-recipe score (decoded via EDA by Chris Deotte) + domain ratios
3. **Tuned LightGBM** — deep, regularized, 3 seeds × 5 folds, averaged
4. **Honest OOF CV** — no LB peeking, shake-up resistant

⏱️ ~45 min CPU (15 LGBM fits). Upvote ⬆ if it helps your run!

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f'{name} not found. Mounted: ' + str(os.listdir('/kaggle/input')))
    return hits[0]

SEEDS = [42, 7, 2026]
N_FOLDS = 5

In [ ]:
train = pd.read_csv(find_file('train.csv'))
test = pd.read_csv(find_file('test.csv'))
orig = pd.read_csv(find_file('EV_Adoption_and_Range_Anxiety_Dataset.csv'))
print('train', train.shape, '| test', test.shape, '| original', orig.shape)

# Original data has NaNs in 3 columns — drop them (534/10000 rows)
orig = orig.dropna(subset=['Annual_Income_USD', 'Daily_Commute_km', 'Environmental_Concern_Level']).reset_index(drop=True)
print('original after dropna:', orig.shape)

y = (train['Will_Buy_EV'] == 'Yes').astype(int).values
y_orig = (orig['Will_Buy_EV'] == 'Yes').astype(int).values
print(f'base rate: train {y.mean():.4f} | original {y_orig.mean():.4f}')

## 1️⃣ DGP-aware feature engineering

The decoded recipe: `score = 1.2·income/100k + 0.6·concern + 2·subsidy − 1·med_anx − 3·high_anx`, buy if `score + N(0,1) > 5.5`. Trees discover this anyway — but hand-delivering it plus interaction features frees capacity for the noise structure.

In [ ]:
def engineer(df):
    df = df.copy()
    home = (df['Home_Charging_Possible'] == 'Yes').astype(int)
    subsidy = (df['Subsidy_Available'] == 'Yes').astype(int)
    anx = df['Range_Anxiety_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})

    # Deotte recipe + worry score
    df['recipe_score'] = (1.2*df['Annual_Income_USD']/1e5 + 0.6*df['Environmental_Concern_Level']
                          + 2.0*subsidy - 1.0*(anx == 1) - 3.0*(anx == 2))
    df['worry_score'] = (df['Daily_Commute_km'] - 5*df['Charging_Stations_Near_Home']
                         - 5*df['Charging_Stations_Near_Work'] - 150*home)
    # Deotte interactions
    df['chargers_total'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
    df['income_x_subsidy'] = df['Annual_Income_USD']/1e5 * subsidy
    df['concern_x_subsidy'] = df['Environmental_Concern_Level'] * subsidy
    # Domain ratios
    df['income_per_km'] = df['Annual_Income_USD'] / (df['Daily_Commute_km'] + 1)
    df['income_per_car'] = df['Annual_Income_USD'] / (df['Number_of_Cars_Owned'] + 1)
    df['infra_x_home'] = df['chargers_total'] * home
    df['anx_x_commute'] = anx * df['Daily_Commute_km']

    for c in ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
              'Subsidy_Available', 'Range_Anxiety_Level']:
        df[c] = df[c].astype('category')
    return df

cols = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
        'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
        'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type',
        'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

X_tr = engineer(train[cols])
X_orig = engineer(orig[cols])
X_test = engineer(test[cols])
print(f'{X_tr.shape[1]} features')

## 2️⃣ 3 seeds × 5 folds with original data upweighted

Original rows carry `sample_weight=20` (they are the true DGP rows; synthetic train dilutes them 67:1). Folds are built on synthetic+original jointly, stratified.

In [ ]:
PARAMS = dict(objective='binary', metric='auc', learning_rate=0.05,
              num_leaves=63, min_child_samples=100, feature_fraction=0.7,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0,
              max_bin=511, verbosity=-1)

X_full = pd.concat([X_tr, X_orig], ignore_index=True)
y_full = np.concatenate([y, y_orig])
w_full = np.concatenate([np.ones(len(y)), np.full(len(y_orig), 20.0)])
cat_cols = X_full.select_dtypes('category').columns.tolist()

oof = np.zeros(len(X_full))
test_pred = np.zeros(len(X_test))

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_full)):
        dtrain = lgb.Dataset(X_full.iloc[tr_idx], y_full[tr_idx], weight=w_full[tr_idx],
                             categorical_feature=cat_cols)
        dvalid = lgb.Dataset(X_full.iloc[va_idx], y_full[va_idx], weight=w_full[va_idx],
                             categorical_feature=cat_cols, reference=dtrain)
        model = lgb.train(PARAMS, dtrain, num_boost_round=3000,
                          valid_sets=[dvalid],
                          callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[va_idx] += model.predict(X_full.iloc[va_idx], num_iteration=model.best_iteration) / len(SEEDS)
        test_pred += model.predict(X_test, num_iteration=model.best_iteration) / (len(SEEDS) * N_FOLDS)
    print(f'seed {seed} done | running OOF AUC: {roc_auc_score(y_full, oof):.5f}')

auc = roc_auc_score(y_full, oof)
auc_synth = roc_auc_score(y_full[:len(y)], oof[:len(y)])
print(f'\n🏆 FINAL OOF AUC (full): {auc:.5f}')
print(f'   OOF AUC (synthetic only): {auc_synth:.5f}')

## 3️⃣ Reference points + submission

In [ ]:
# Sanity reference: recipe-only AUC on the same full set
recipe_full = X_full['recipe_score'].values
print(f'recipe-only AUC: {roc_auc_score(y_full, recipe_full):.5f} (ceiling of pure DGP formula)')
print(f'our model AUC:   {auc:.5f}  → model extracts {auc - roc_auc_score(y_full, recipe_full):.5f} beyond the recipe')

np.savez_compressed('oof_original.npz', oof=oof, y_full=y_full, test_pred=test_pred)

sub = pd.DataFrame({'id': test['id'], 'Will_Buy_EV': test_pred})
sub.to_csv('submission.csv', index=False)
print('submission.csv saved |', sub.shape, '| mean', round(sub['Will_Buy_EV'].mean(), 4))
sub.head()

## 🎯 Notes
- **Trust the OOF**: this comp's public LB is a tiny noisy slice; the private split rewards honest CV.
- Original-data upweighting is the single biggest lever here (+0.002–0.004 CV in sibling playgrounds).
- Next levers: add XGBoost/CatBoost members to the blend, per-seed hyper jitter, or quantile-band calibration if you understand the public split risk.

If this got you a personal best, an upvote ⬆ keeps the updates coming!